# 01 — PyTorch baselines (FP32)

Runs:
- FP32 (CUDA)
- FP16 (CUDA)

Optionally without reduced bit depth.

In [4]:
from pathlib import Path
import sys, os

# ---- Path setup (adjust if your repo layout differs) ----
PROJECT_ROOT = Path("..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [5]:
# Purpose: Establish PyTorch FP32 baseline (and optional FP16 baseline) on ImageNet val.
import os
import json
from pathlib import Path
import pandas as pd

# project imports
from config import ExperimentConfig, with_overrides
from runner import run_experiment
from utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)

In [6]:
# Baselines (PyTorch backend only)
base = ExperimentConfig(
    backend="pytorch",
    device="cuda",             
    batch_size=1,
    input_quant_bits=8,        
    seed=42,
    num_eval_batches=500
)


In [7]:
cfg32 = with_overrides(base, model_precision="fp32")

payload, tracker = run_experiment(cfg32, save_results_flag=True)

# quick peek
print(payload["run_id"], payload["status"], payload["results"].get("top1_acc"))

[data] Filtered to 5000 samples across 100 classes.
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 3.09 ms/batch
  Batch [50/500] Top-1: 95.00% | Top-5: 100.00% | Infer: 2.99 ms/batch
  Batch [60/500] Top-1: 96.67% | Top-5: 100.00% | Infer: 2.91 ms/batch
  Batch [70/500] Top-1: 97.50% | Top-5: 100.00% | Infer: 2.92 ms/batch
  Batch [80/500] Top-1: 94.00% | Top-5: 98.00% | Infer: 2.93 ms/batch
  Batch [90/500] Top-1: 95.00% | Top-5: 98.33% | Infer: 3.06 ms/batch
  Batch [100/500] Top-1: 94.29% | Top-5: 98.57% | Infer: 3.11 ms/batch
  Batch [110/500] Top-1: 95.00% | Top-5: 98.75% | Infer: 3.10 ms/batch
  Batch [120/500] Top-1: 93.33% | Top-5: 98.89% | Infer: 3.12 ms/batch
  Batch [130/500] Top-1: 94.00% | Top-5: 99.00% | Infer: 3.07 ms/batch
  Batch [140/500] Top-1: 92.73% | Top-5: 99.09% | Infer: 2.99 ms/batch
  Batch [150/500] Top-1: 91.67% | Top-5: 99.17% | 

In [8]:
# Always load from disk (the source of truth)
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter to just this notebook’s scope (pytorch + no input quant)
df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp32"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.backend",
    "cfg.device",
    "cfg.batch_size",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.device"])

,run_id,cfg.backend,cfg.device,cfg.batch_size,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
0,resnet18_pytorch_fp32_in8b_cuda_bs1,pytorch,cuda,1,fp32,8,83.829787,96.595745,3.093748,323.232546,470


In [9]:
cfg16 = with_overrides(base, model_precision="fp16")

payload, tracker = run_experiment(cfg16, save_results_flag=True)

# quick peek
print(payload["run_id"], payload["status"], payload["results"].get("top1_acc"))

[data] Filtered to 5000 samples across 100 classes.
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 2.34 ms/batch
  Batch [50/500] Top-1: 95.00% | Top-5: 100.00% | Infer: 2.74 ms/batch
  Batch [60/500] Top-1: 96.67% | Top-5: 100.00% | Infer: 2.71 ms/batch
  Batch [70/500] Top-1: 97.50% | Top-5: 100.00% | Infer: 2.66 ms/batch
  Batch [80/500] Top-1: 94.00% | Top-5: 98.00% | Infer: 2.77 ms/batch
  Batch [90/500] Top-1: 95.00% | Top-5: 98.33% | Infer: 2.78 ms/batch
  Batch [100/500] Top-1: 94.29% | Top-5: 98.57% | Infer: 2.64 ms/batch
  Batch [110/500] Top-1: 95.00% | Top-5: 98.75% | Infer: 2.68 ms/batch
  Batch [120/500] Top-1: 93.33% | Top-5: 98.89% | Infer: 2.72 ms/batch
  Batch [130/500] Top-1: 94.00% | Top-5: 99.00% | Infer: 2.67 ms/batch
  Batch [140/500] Top-1: 92.73% | Top-5: 99.09% | Infer: 2.61 ms/batch
  Batch [150/500] Top-1: 91.67% | Top-5: 99.17% | 

In [10]:
# Always load from disk (the source of truth)
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter to just this notebook’s scope (pytorch + no input quant)
df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp16"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.backend",
    "cfg.device",
    "cfg.batch_size",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.device"])

,run_id,cfg.backend,cfg.device,cfg.batch_size,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps,res.total_samples
0,resnet18_pytorch_fp16_in8b_cuda_bs1,pytorch,cuda,1,fp16,8,83.829787,96.595745,2.885167,346.600371,470


In [11]:
runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

df_baselines = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.input_quant_bits"] == 8)
    & (df["cfg.model_precision"].isin(["fp32", "fp16"]))
].copy()

df_baselines[[
    "run_id",
    "cfg.model_precision",
    "res.infer_ms_avg",
    "res.infer_ms_min",
    "res.infer_ms_max",
]].sort_values("cfg.model_precision")

,run_id,cfg.model_precision,res.infer_ms_avg,res.infer_ms_min,res.infer_ms_max
0,resnet18_pytorch_fp16_in8b_cuda_bs1,fp16,2.885167,1.041826,6.366387
1,resnet18_pytorch_fp32_in8b_cuda_bs1,fp32,3.093748,1.236312,6.725033
